<a href="https://colab.research.google.com/github/serelk/nebius-academy-course/blob/main/week3/task-6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community
!pip install -U langchain-huggingface
!pip install -qU langchain-community faiss-cpu

In [17]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import os
from openai import OpenAI
from google.colab import userdata
import pandas as pd
from datasets import load_dataset

In [15]:


dataset = load_dataset("PatronusAI/financebench")
df = dataset["train"].to_pandas()

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
embedding_dim = 384  # Dimension for 'BAAI/bge-small-en-v1.5'
# index = faiss.IndexFlatL2(embedding_dim)

In [6]:
drive_path = '/content/drive/MyDrive/faiss_indexes'


drive_saved_path = os.path.join(drive_path, "my_faiss_index")
loaded_vector_store = FAISS.load_local(
    drive_saved_path, embeddings, allow_dangerous_deserialization=True
)
print(f"FAISS index loaded from {drive_saved_path}/")

FAISS index loaded from /content/drive/MyDrive/faiss_indexes/my_faiss_index/


In [7]:
client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=userdata.get('NEBIUS_API_KEY')
)

In [8]:
retriever = loaded_vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 100,
        "lambda_mult": 0.5,
    },
)

In [ ]:
question_1 = "Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why."

retrieved_docs = retriever.invoke(question_1)
print(len(retrieved_docs))
for doc in retrieved_docs:
    print(doc.metadata["page"])



In [ ]:
question_1 = "Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why."

retrieved_docs = retriever.invoke(question_1)
print(retrieved_docs[0].page_content)


In [9]:
# evidence = """ evidence_text': 'Consolidated Balance Sheets Corning Incorporated and Subsidiary Companies December 31, (in millions, except share and per share amounts) 2022 2021 Assets Current assets: Cash and cash equivalents $ 1,671 $ 2,148 Trade accounts receivable, net of doubtful accounts - $40 and $42 1,721 2,004 Inventories (Note 5) 2,904 2,481 Other current assets (Notes 10 and 14) 1,157 1,026 Total current assets 7,453 7,659 Property, plant and equipment, net of accumulated depreciation - $14,147 and $13,969 (Note 8) 15,371 15,804 Goodwill, net (Note 9) 2,394 2,421 Other intangible assets, net (Note 9) 1,029 1,148 Deferred income taxes (Note 7) 1,073 1,066 Other assets (Notes 10 and 14) 2,179 2,056 Total Assets $ 29,499 $ 30,154 Liabilities and Equity Current liabilities: Current portion of long-term debt and short-term borrowings (Note 11) $ 224 $ 55 Accounts payable 1,804 1,612 Other accrued liabilities (Notes 10 and 13) 3,147 3,139 Total current liabilities 5,175 4,806 Long-term debt (Note 11) 6,687 6,989 Postretirement benefits other than pensions (Note 12) 407 622 Other liabilities (Notes 10 and 13) 4,955 5,192 Total liabilities 17,224 17,609 Commitments and contingencies (Note 13) Shareholders equity (Note 16): Common stock Par value $0.50 per share; Shares authorized 3.8 billion; Shares issued: 1.8 billion and 1.8 billion 910 907 Additional paid-in capital common stock 16,682 16,475 Retained earnings 16,778 16,389 Treasury stock, at cost; Shares held: 977 million and 970 million (20,532) (20,263) Accumulated other comprehensive loss (1,830) (1,175) Total Corning Incorporated shareholders equity 12,008 12,333 Non-controlling interest 267 212 Total equity 12,275 12,545 Total Liabilities and Equity $ 29,499 $ 30,154 The accompanying notes are an integral part of these consolidated financial statements. 60"""

def answer_with_rag(question: str):
    # question_1 = "What are Corning's current assets and current liabilities for FY2022, and based on this, does Corning have positive working capital? If working capital is not a useful or relevant metric for this company, then please state that and explain why."
    retrieved_docs = retriever.invoke(question)
    context = retrieved_docs[0].page_content

    print("--- Context sent to LLM ---")
    print(context)
    print("-------------------------")

    system_prompt = """
You are a helpful technical assistant tasked with answering financial questions based on provided financial statements.
Answer only using the provided context. If the context is not enough, say you don't know.
"""

    user_prompt = f"""
Context:
{context}

Question:
{question}
"""

    try:
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.3-70B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": user_prompt.strip()}
            ],
            temperature=0
        )

        return response.choices[0].message.content
    except NameError:
        return "Error: 'client' for chat completions is not defined. Please ensure your LLM client is initialized."


question_1 = "Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why."

answer_with_rag(question_1)

--- Context sent to LLM ---
(1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September 9, 2020.  Refer to Note 3 (HSGTransactions and Acquisitions) in the notes to the consolidated financial statements for additional information.(2) Depreciation expense for Corning’s reportable segments and Hemlock and Emerging Growth Businesses includes an allocation of depreciation of corporate property not specifically identifiable to asegment.(3) Research, development and engineering expenses include direct project spending that is identifiable to a segment.(4) Income tax (provision) benefit reflects a tax rate of 21%.(5) Segment assets include inventory, accounts receivable, property, plant and equipment, net of accumulated depreciation, and associated equity companies.
 102
-------------------------


"I don't know. The provided context does not include the necessary financial data, such as current assets and current liabilities, to determine Corning's working capital for FY2022. Additionally, the context only provides general information about Corning's financial statements and segment reporting, but does not include specific financial data or balances."

In [ ]:
import numpy as np
pd.set_option('display.max_colwidth', None)
filtered_df = df.where(df.question_type == "domain-relevant").dropna().sort_values(by="financebench_id").head(5)

# Display evidence_page_num and evidence_text for each entry
for index, row in filtered_df.iterrows():
    print(f"Financebench ID: {row['financebench_id']}")
    print(f"Question: {row['question']}")
    print(f"Expected_Answer: {row['answer']}")
    # print(f"  Type of evidence: {type(row['evidence'])}") # Added for debugging
    # print(f"  Raw Evidence Content: {row['evidence']}") # Added for debugging

    llm_answer = answer_with_rag(row['question'])
    print(f"LLM_Answer: {llm_answer}")
    evidence_data = row['evidence']
    if isinstance(evidence_data, np.ndarray):
        evidence_data = evidence_data.tolist() # Convert numpy array to list

    if isinstance(evidence_data, list):
        for evidence_item in evidence_data:
            page_num = evidence_item.get('evidence_page_num')
            evidence_text = evidence_item.get('evidence_text')
            print(f"  Evidence Page Num: {page_num}")
            # print(f"  Evidence Text: {evidence_text}")
    else:
        print("  No evidence found or evidence format is unexpected.")
    print("---\n")